In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.pandas as ps
import pyspark.sql.functions as F
import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [3]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer
from hypex.utils import SparkSessionCalculator
from hypex.config import DatasetConfig

In [4]:
import os
import time
from pyspark.sql import SparkSession
from hypex.utils.spark_config import SparkSessionCalculator

# --- 1. Настройки окружения для macOS (Важно!) ---
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация для калькулятора ---
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048

# Параметры для калькулятора (на основе вашего датасета)
n_rows = 10_000
n_columns = 7
n_categorical = 1

# --- 3. Создание и использование калькулятора ---
calculator = SparkSessionCalculator(
    data_size_bytes=n_rows * n_columns * 8,  # Примерная оценка
    num_columns=n_columns,
    num_categorical_columns=n_categorical,
    target_executor_cores=CORES_PER_EXECUTOR,
    target_executor_memory_gb=MEMORY_PER_EXECUTOR_MB / 1024
)

# Получаем оптимальные настройки
optimal_settings = calculator.calculate_optimal_settings()

print("\n📊 Оптимальные настройки от калькулятора:")
print(f"  Executor instances: {optimal_settings.executor_instances}")
print(f"  Executor cores: {optimal_settings.executor_cores}")
print(f"  Executor memory: {optimal_settings.executor_memory}")
print(f"  Shuffle partitions: {optimal_settings.shuffle_partitions}")

# --- 4. Создание Spark-сессии с настройками калькулятора ---
MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"
print(f"\n🚀 Запуск в режиме: {MASTER_URL}")

builder = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Настройки драйвера
    .config("spark.driver.memory", "2g")
    # Настройки executor'ов из калькулятора
    .config("spark.executor.memory", f"{MEMORY_PER_EXECUTOR_MB}m")
    .config("spark.executor.cores", str(CORES_PER_EXECUTOR))
    .config("spark.executor.instances", str(NUM_EXECUTORS))
    # Дополнительные настройки из калькулятора
    .config("spark.memory.fraction", str(optimal_settings.memory_fraction))
    .config("spark.sql.shuffle.partitions", str(optimal_settings.shuffle_partitions))
    .config("spark.serializer", optimal_settings.serializer)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true")
)

sp_s = builder.getOrCreate()
sp_s.sparkContext.setLogLevel("WARN")

# --- 5. Проверка конфигурации ---
print(f"\n✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")
print(f"Shuffle Partitions: {sp_s.conf.get('spark.sql.shuffle.partitions')}")

# Проверка количества экзекуторов
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 6. Тест на распределение ---
def print_executor_info(iterator):
    import os
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

df = sp_s.range(0, 10, 1, 4)
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# sp_s.stop()  # Раскомментируйте, если нужно остановить сессию


📊 Оптимальные настройки от калькулятора:
  Executor instances: 4
  Executor cores: 4
  Executor memory: 4g
  Shuffle partitions: 10

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/09/08 13:07:42 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.77.138 instead (on interface wlo1)
26/09/08 13:07:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 13:07:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2048m
Shuffle Partitions: 10


📊 Активных экзекуторов (проверка через RDD): 2



🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 540732
Executor ID: Driver/Local, PID: 540738
Executor ID: Driver/Local, PID: 540802
Executor ID: Driver/Local, PID: 540801


In [5]:
n_rows = 10000
n = n_rows
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [6]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
)

In [7]:
from hypex.config import MatchingConfig
MatchingConfig.FAISS_FIT_MODE = "full"
MatchingConfig.FAISS_CHUNK_SIZE = 512

In [8]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=True,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=10,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
# print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
# print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

🚀 Запуск пайплайна Matching...


26/09/08 13:08:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
/home/eric/HypEx/HypEx/hypex/dataset/backends/spark_backend.py:94: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(
/home/eric/HypEx/HypEx/hypex/dataset/backends/spark_backend.py:94: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(
/home/eric/HypEx/HypEx/hypex/dataset/backends/spark_backend.py:94: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if a

✅ Выполнено успешно!


In [9]:
result_data.resume

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,-0.08,0.22,0.72,-0.51,0.35
ATC,-0.02,0.22,0.94,-0.44,0.41
ATE,-0.04,0.21,0.85,-0.46,0.38


In [10]:
from hypex.utils import FaissIndexStorage

FaissIndexStorage.cleanup()

In [12]:
# 1. Clear DataFrame and SQL Catalog level caches
sp_s.catalog.clearCache()

# 2. Extract the internal Java Spark Context map of persistent RDDs
persistent_rdds = sp_s.sparkContext._jsc.getPersistentRDDs()

# 3. Forcefully unpersist every remaining RDD block sequentially
for rdd_id in list(persistent_rdds.keys()):
    persistent_rdds.get(rdd_id).unpersist(True)  # True forces a synchronous blocking wipe

print("All DataFrame and RDD caches have been forced out of storage.")


All DataFrame and RDD caches have been forced out of storage.


In [12]:
sp_s

In [16]:
dataset

,treatment,feat_num_1,feat_num_2,feat_cat,target
0,0,9.622831,-0.98733,B,101.868128
1,1,13.308879,-3.137177,A,100.832645
2,0,10.25063,-1.842763,B,99.83202
3,0,9.222047,-0.324709,C,83.409315
4,0,12.180395,-3.969011,B,124.800878
...,...,...,...,...,...
9995,0,5.627062,-3.718456,A,115.432218
9996,0,12.305129,-0.57642,A,104.331978
9997,1,5.967907,-1.544364,A,98.239837
9998,0,2.900358,-1.51496,A,91.895542


In [10]:
sp_s.stop()